# RAG Pipeline Setup

## Setup: Install Libraries

In [2]:
pip install PyMuPDF sentence-transformers numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 59.1 MB/s eta 0:00:00


## Setup: Import Dependencies

In [3]:
import fitz # PyMuPDF
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## Configuration

## Gemini API Setup

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.
In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then pass the key to the SDK:

In [4]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

try:
    GOOGLE_API_KEY=userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("Gemini API configured successfully.")
except userdata.SecretNotFoundError:
    print("GOOGLE_API_KEY not found in Colab secrets. Please add it to access the Gemini API.")
except Exception as e:
    print(f"An error occurred during Gemini API configuration: {e}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini API configured successfully.


Before you can make any API calls, you need to initialize the Generative Model.

In [7]:
# Initialize the Gemini API
# You can choose a different model if needed, e.g., 'gemini-pro'
try:
    gemini_model = genai.GenerativeModel('gemini-2.5-flash')
    print("Gemini model initialized: gemini-2.5-flash")
except Exception as e:
    print(f"Failed to initialize Gemini model: {e}")
    gemini_model = None

Gemini model initialized: gemini-2.5-flash


## Data Loading and Preprocessing

### Upload PDF and Extract Text

In [8]:
from google.colab import files
import io

def extract_text_from_pdf(pdf_file):
    """Extracts text from a PDF file using PyMuPDF."""
    doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    text = ""
    for page_num in range(doc.page_count):
        page = doc[page_num]
        text += page.get_text()
    doc.close()
    return text

# Upload PDF file
print("Please upload a PDF file:")
uploaded = files.upload()

if uploaded:
    for filename, data in uploaded.items():
        print(f'Uploaded file: {filename}')
        pdf_file = io.BytesIO(data)
        extracted_text = extract_text_from_pdf(pdf_file)

        print("\n--- First 1000 characters of extracted text ---")
        print(extracted_text[:1000])
        break # Process only the first uploaded file
else:
    print("No file uploaded.")

Please upload a PDF file:


Saving Sound.pdf to Sound (1).pdf
Uploaded file: Sound (1).pdf

--- First 1000 characters of extracted text ---
Everyday we hear sounds from various
sources like humans, birds, bells, machines,
vehicles, televisions, radios etc. Sound is a
form of energy which produces a sensation
of hearing in our ears. There are also other
forms of energy like mechanical energy, light
energy, etc. We have talked about mechanical
energy in the previous chapters. You have
been taught about conservation of energy,
which states that we can neither create nor
destroy energy. We  can just change it from
one form to another. When you clap, a sound
is produced. Can you produce sound without
utilising your energy? Which form of energy
did you use to produce sound? In this
chapter we are going to learn how sound is
produced and how it is transmitted through
a medium and received by our ears.
11.1 Production of Sound
Activity _____________11.1
•
Take a tuning fork and set it vibrating
by striking its prong on a

### Text Chunking

In [9]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Splits text into chunks with specified size and overlap."""
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
        if i + chunk_size >= len(words):
            break
        i += chunk_size - overlap
    return chunks

if 'extracted_text' in locals() and extracted_text:
    text_chunks = chunk_text(extracted_text, chunk_size=500, overlap=50)
    print(f"Total number of chunks: {len(text_chunks)}")
    print("\n--- First chunk preview ---")
    print(text_chunks[0][:500] + '...') # Display first 500 characters of the first chunk
else:
    print("No text extracted. Please upload a PDF and run the previous cell.")

Total number of chunks: 13

--- First chunk preview ---
Everyday we hear sounds from various sources like humans, birds, bells, machines, vehicles, televisions, radios etc. Sound is a form of energy which produces a sensation of hearing in our ears. There are also other forms of energy like mechanical energy, light energy, etc. We have talked about mechanical energy in the previous chapters. You have been taught about conservation of energy, which states that we can neither create nor destroy energy. We can just change it from one form to another. Wh...


## Embedding Generation

In [10]:
if 'text_chunks' in locals() and text_chunks:
    print("Loading SentenceTransformer model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    print("Model loaded.")

    print("Generating embeddings for text chunks...")
    chunk_embeddings = model.encode(text_chunks, show_progress_bar=True)
    print("Embeddings generated.")

    # Store chunks and their embeddings
    embedded_chunks = []
    for i, chunk in enumerate(text_chunks):
        embedded_chunks.append({"text": chunk, "embedding": chunk_embeddings[i]})

    print(f"\nShape of embeddings: {chunk_embeddings.shape}")
    print("\n--- Sample embedded chunk entry ---")
    sample_entry = embedded_chunks[0]
    print(f"Text: {sample_entry['text'][:200]}...") # Print first 200 chars of text
    print(f"Embedding (first 5 values): {sample_entry['embedding'][:5]}...")
else:
    print("Text chunks not found. Please ensure PDF upload and chunking cells were executed successfully.")

Loading SentenceTransformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.
Generating embeddings for text chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated.

Shape of embeddings: (13, 384)

--- Sample embedded chunk entry ---
Text: Everyday we hear sounds from various sources like humans, birds, bells, machines, vehicles, televisions, radios etc. Sound is a form of energy which produces a sensation of hearing in our ears. There ...
Embedding (first 5 values): [ 0.01654705 -0.0636483   0.00018998  0.00313182 -0.04454342]...


## RAG Pipeline Implementation

## LLM Integration for Answering

## Sample Image Metadata

In [11]:
image_metadata = [
    {
        "id": "CompressionAndRefraction",
        "title": "Compression and Refraction",
        "keywords": ["sound", "wave", "compression", "refraction", "diagram"],
        "description": "A diagram illustrating the concepts of compression and refraction in sound waves."
    },
    {
        "id": "MusicalInstrumentsVibrationChart",
        "title": "Musical Instruments Vibration Chart",
        "keywords": ["musical instruments", "vibration", "sound production", "chart"],
        "description": "A chart showing how different musical instruments produce sound through vibration."
    },
    {
        "id": "ReflectionOfSound",
        "title": "Reflection of Sound",
        "keywords": ["sound", "reflection", "echo", "diagram"],
        "description": "A diagram explaining the phenomenon of sound reflection, often leading to echoes."
    },
    {
        "id": "SchoolBellVibration",
        "title": "School Bell Vibration",
        "keywords": ["school bell", "vibration", "sound production", "example"],
        "description": "An image or diagram illustrating the vibration of a school bell to produce sound."
    },
    {
        "id": "VibrationOfRubberBand",
        "title": "Vibration of Rubber Band",
        "keywords": ["rubber band", "vibration", "sound production", "experiment"],
        "description": "An image demonstrating how a vibrating rubber band produces sound, often used in simple experiments."
    },
    {
        "id": "VocalCordsDiagram",
        "title": "Vocal Cords Diagram",
        "keywords": ["vocal cords", "voice", "sound production", "human anatomy", "diagram"],
        "description": "A diagram of human vocal cords, illustrating their role in producing speech and sound."
    }
]

display(image_metadata)

[{'id': 'CompressionAndRefraction',
  'title': 'Compression and Refraction',
  'keywords': ['sound', 'wave', 'compression', 'refraction', 'diagram'],
  'description': 'A diagram illustrating the concepts of compression and refraction in sound waves.'},
 {'id': 'MusicalInstrumentsVibrationChart',
  'title': 'Musical Instruments Vibration Chart',
  'keywords': ['musical instruments',
   'vibration',
   'sound production',
   'chart'],
  'description': 'A chart showing how different musical instruments produce sound through vibration.'},
 {'id': 'ReflectionOfSound',
  'title': 'Reflection of Sound',
  'keywords': ['sound', 'reflection', 'echo', 'diagram'],
  'description': 'A diagram explaining the phenomenon of sound reflection, often leading to echoes.'},
 {'id': 'SchoolBellVibration',
  'title': 'School Bell Vibration',
  'keywords': ['school bell', 'vibration', 'sound production', 'example'],
  'description': 'An image or diagram illustrating the vibration of a school bell to produce 

In [12]:
if 'model' not in locals() or not isinstance(model, SentenceTransformer):
    print("Loading SentenceTransformer model for image metadata embeddings...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    print("Model loaded.")

image_descriptions = []
for item in image_metadata:
    # Combine description and keywords for embedding
    combined_text = item['description'] + " " + ", ".join(item['keywords'])
    image_descriptions.append(combined_text)

print("Generating embeddings for image metadata descriptions...")
image_embeddings = model.encode(image_descriptions, show_progress_bar=True)

embedded_image_metadata = []
for i, item in enumerate(image_metadata):
    embedded_image_metadata.append({
        "id": item["id"],
        "embedding": image_embeddings[i]
    })

print("Embeddings generated for image metadata.")
print(f"\nShape of image embeddings: {image_embeddings.shape}")
print("\n--- Sample embedded image entry ---")
sample_image_entry = embedded_image_metadata[0]
print(f"ID: {sample_image_entry['id']}")
print(f"Embedding (first 5 values): {sample_image_entry['embedding'][:5]}...")

Generating embeddings for image metadata descriptions...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated for image metadata.

Shape of image embeddings: (6, 384)

--- Sample embedded image entry ---
ID: CompressionAndRefraction
Embedding (first 5 values): [-0.04303913 -0.01627243 -0.04064489 -0.03040579 -0.07637408]...


In [13]:
def find_similar_image(query_or_answer, model, embedded_image_metadata):
    """Finds the most similar image to a given query or LLM answer."""
    if not embedded_image_metadata:
        print("Error: No embedded image metadata available.")
        return None, 0.0

    # Encode the query or answer
    query_embedding = model.encode(query_or_answer)

    # Extract only the embeddings from the list of dictionaries
    image_vectors = np.array([item["embedding"] for item in embedded_image_metadata])

    # Compute cosine similarity
    similarities = cosine_similarity(query_embedding.reshape(1, -1), image_vectors)[0]

    # Get the index of the highest similarity score
    best_match_index = np.argmax(similarities)

    # Get the corresponding image ID and score
    best_image_id = embedded_image_metadata[best_match_index]["id"]
    best_score = similarities[best_match_index]

    return best_image_id, best_score

In [14]:
# --- Demo Image Retrieval ---
if 'model' in locals() and 'embedded_image_metadata' in locals() and embedded_image_metadata:
    # Example 1: User query
    user_query_image = "What equipment is used for recording sound?"
    best_image_id_query, best_score_query = find_similar_image(user_query_image, model, embedded_image_metadata)
    print(f"\nUser Query for Image: {user_query_image}")
    print(f"Best matching image (Query): {best_image_id_query} (Similarity: {best_score_query:.4f})")

    # Example 2: Hypothetical LLM answer
    llm_answer_image = "Microphones are essential for capturing vocal and instrumental sounds in a studio environment."
    best_image_id_llm, best_score_llm = find_similar_image(llm_answer_image, model, embedded_image_metadata)
    print(f"\nLLM Answer for Image: {llm_answer_image}")
    print(f"Best matching image (LLM Answer): {best_image_id_llm} (Similarity: {best_score_llm:.4f})")
else:
    print("Dependencies for image retrieval not met. Please ensure embedding cells were executed successfully.")


User Query for Image: What equipment is used for recording sound?
Best matching image (Query): MusicalInstrumentsVibrationChart (Similarity: 0.3373)

LLM Answer for Image: Microphones are essential for capturing vocal and instrumental sounds in a studio environment.
Best matching image (LLM Answer): VocalCordsDiagram (Similarity: 0.3904)


In [16]:
def generate_llm_response(query, context_chunks, llm_model):
    """Generates a response using an LLM based on a query and provided context."""
    if not llm_model:
        return "Error: Gemini model not initialized."

    context = "\n".join([chunk['text'] for chunk in context_chunks])

    prompt = f"""Answer ONLY using the context below. If the answer is not in the context, state that you cannot answer from the provided information.

Context:
{context}

Question: {query}
Answer:"""

    try:
        response = llm_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error generating response from LLM: {e}"

In [20]:
# --- Demo LLM Answering ---
if 'model' in locals() and 'embedded_chunks' in locals() and embedded_chunks and 'gemini_model' in locals() and gemini_model:
    # Use the same user_query from similarity search demo
    user_query = "How does the sound produced by a vibrating object in a medium reach your ear?"
    print(f"User Query for LLM: {user_query}")

    # Assuming top_3_chunks was generated successfully in the previous cell
    # If not, we need a fallback or re-run the previous cell
    if 'top_3_chunks' not in locals() or not top_3_chunks:
        print("Running similarity search to get chunks for LLM...")
        top_3_chunks = find_similar_chunks(user_query, model, embedded_chunks, top_n=3)

    if top_3_chunks:
        print("\n--- Context Provided to LLM ---")
        for i, chunk in enumerate(top_3_chunks):
            print(f"Chunk {i+1} (Similarity: {chunk['similarity']:.4f}):\n{chunk['text'][:200]}...")

        llm_answer = generate_llm_response(user_query, top_3_chunks, gemini_model)
        print("\n--- Final LLM Answer ---")
        print(llm_answer)
    else:
        print("No relevant chunks found to provide to the LLM.")
else:
    print("Dependencies for LLM answering not met. Please ensure previous cells (PDF upload, chunking, embedding, similarity search, Gemini API setup) were executed successfully.")

User Query for LLM: How does the sound produced by a vibrating object in a medium reach your ear?

--- Context Provided to LLM ---
Chunk 1 (Similarity: 0.0161):
= 346 m s–1 × 2 s = 692 m In 2 s sound has to travel twice the distance between the cliff and the person. Hence, the distance between the cliff and the person = 692 m/2 = 346 m. Q Horn Megaphone Fig 1...
Chunk 2 (Similarity: 0.0035):
of compressions and rarefactions passing a fixed point per unit time. Objects of different sizes and conditions vibrate at different frequencies to produce sounds of different pitch. The magnitude of ...
Chunk 3 (Similarity: -0.0128):
travels through the medium and not the particles of the medium. A wave is a disturbance that moves through a medium when the particles of the medium set neighbouring particles into motion. They in tur...

--- Final LLM Answer ---
Sound is produced by vibrating objects. When an object vibrates, it sets the particles of the surrounding medium (which can be solid, liquid

In [23]:
def find_similar_chunks(query, model, embedded_chunks, top_n=3):
    """Finds the top N most similar chunks to a given query."""
    if not embedded_chunks:
        print("Error: No embedded chunks available.")
        return []

    query_embedding = model.encode(query)

    # Extract only the embeddings from the list of dictionaries
    chunk_vectors = np.array([item["embedding"] for item in embedded_chunks])

    # Compute cosine similarity between query and all chunk embeddings
    similarities = cosine_similarity(query_embedding.reshape(1, -1), chunk_vectors)[0]

    # Get indices of top N similar chunks
    top_indices = similarities.argsort()[-top_n:][::-1]

    top_chunks = []
    for i in top_indices:
        top_chunks.append({
            "text": embedded_chunks[i]["text"],
            "similarity": similarities[i]
        })
    return top_chunks

# --- Demo Similarity Search ---
if 'model' in locals() and 'embedded_chunks' in locals() and embedded_chunks:
    user_query = "How does the sound produced by a vibrating object in a medium reach your ear?"
    print(f"User Query: {user_query}")

    # Find and print top 3 similar chunks
    top_3_chunks = find_similar_chunks(user_query, model, embedded_chunks, top_n=3)

    print("\n--- Top 3 Similar Chunks ---")
    for i, chunk in enumerate(top_3_chunks):
        print(f"Chunk {i+1} (Similarity: {chunk['similarity']:.4f}):\n{chunk['text'][:500]}...\n")
else:
    print("Model or embedded chunks not found. Please ensure all previous cells were executed successfully.")

User Query: How does the sound produced by a vibrating object in a medium reach your ear?

--- Top 3 Similar Chunks ---
Chunk 1 (Similarity: 0.6260):
(R) in the medium. Reprint 2025-26 SOUND 129 propagates through the medium. Compression is the region of high pressure and rarefaction is the region of low pressure. Pressure is related to the number of particles of a medium in a given volume. More density of the particles in the medium gives more pressure and vice versa. Thus, propagation of sound can be visualised as propagation of density variations or pressure variations in the medium. uestion 1. How does the sound produced by a vibrating ob...

Chunk 2 (Similarity: 0.5822):
travels through the medium and not the particles of the medium. A wave is a disturbance that moves through a medium when the particles of the medium set neighbouring particles into motion. They in turn produce similar motion in others. The particles of the medium do not move forward themselves, but the disturbance